# ***INSTALACIÓN LIBRERIAS***

Si queremos ejecutar todo el codigo de manera seguida recomiendo instalarse todas las librerias en el siguiente paso.
Por otro lado si desea ejecturar los apartados por separados y en cualquier orden, puede hacerlo sin problema.

In [ ]:
import re #importamos la libreria "re" que la necesitamos para las expresiones regulares
import os #importamos la libreria "os" que la utlizamos para las operaciones del sistema
from google.colab import files #importamos de "google.colab" la parte de "files" para asi poder descargar archivos
import ipaddress #importamos la librería "ipaddress" para validar los dos tipos de ip: IPv4 e IPv6
from urllib.parse import urlparse, unquote #Importamos de la libreria "urlparse, unquote" la funcion "urllib.parse" para analziar las urls y decodificar caracteres especiales
import datetime #Importamos esta libreria para generar fechas y horas en la parte de los informes
import socket #Importamos esta libreria para hacer las consultas DNS y resolver las IPs y los dominios
import json #Importamos esta libreria para el informe RDAP
import request #Importamos esta libreria para hacer peticiones HTTP a sevicios web

# ***GUÍA DE USO***

En este cuaderno tenemos cinco pasos principales para trabajar con Indicadores de Compromiso. Cada sección tiene su propio menú interactivo donde podemos elegir entre analizar un IOC individual o subir un archivo .txt con varios IOC.

Una vez empezemos a ejecutes cualquier sección, veremos un menú con tres opciones: la opción 1 es para introducir un IOC sueltos, la opción 2 para introducir archivos.txt (te pedirá que selecciones el .txt desde la propia memoria de tu ordenador y luego elegir si ver los resultados en pantalla o descargarlos), y la opción 3 para salir. Los archivos que generas y decides descargar en el ordenadorse nombran automáticamente para que puedas usarlos en el siguiente (tipo `archivo_desofuscado.txt`, `archivo_normalizado.txt`, etc.).

La idea de ejecución de este codigo es seguir el orden lógico: primero desofuscar, luego normalizar para que tengan formato estándar, después clasificarlos para saber si son IPs, URLs o dominios, hacer resolución DNS para ver sus registros, y finalmente consultar RDAP para información de registro y propiedad. Cada paso valida que lo anterior esté bien hecho, así que si te salta algún error, probablemente necesites volver al paso anterior.

Ademas para el tema de subir archivos.txt simplemente hace falta puslar en el boton izquierdo que te muestra al introducir el valor 2. Si por el contrario pulsamos el boton de la derecha se cancelaria la subida de archivo.

# ***DESOFUSCAR***

In [ ]:
import re
import os
from google.colab import files
import ipaddress

# Validación de IOCs
def es_ioc_valido(texto):
    texto = texto.strip()

    # Verificar si es IP
    try:
        ipaddress.ip_address(texto)
        return True
    except ValueError:
        pass

    # Verificar si es dominio simple (ejemplo.com)
    if re.match(r'^[a-zA-Z0-9]([a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?(\.[a-zA-Z]{2,})+$', texto):
        return True

    # Verificar si es URL con protocolo
    if re.match(r'^(http|https|ftp)://[^\s/$.?#].[^\s]*$', texto):
        return True

    # Verificar si es dominio con ruta (ejemplo.com/path)
    if re.match(r'^[a-zA-Z0-9]([a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?(\.[a-zA-Z]{2,})+[/\w\.-]*$', texto):
        return True

    return False

# Desofuscación
def desofuscar_indicadores(texto):
    patrones = [
        (r'\[\.\]', '.'),
        (r'\[dot\]', '.'),
        (r'\(dot\)', '.'),
        (r'\[ \]', ''),
        (r'\[\]', ''),
        (r'hxxps://', 'https://'),
        (r'hxxp://', 'http://'),
        (r'h\*\*p://', 'http://'),
        (r'htxp://', 'http://'),
        (r'\[://\]', '://'),
        (r'\[:\]', ':'),
        (r'//\]', '//'),
    ]
    texto_desofuscado = texto
    for patron, reemplazo in patrones:
        texto_desofuscado = re.sub(patron, reemplazo, texto_desofuscado, flags=re.IGNORECASE)

    return texto_desofuscado

# Procesar archivo
def procesar_archivo_colab(nombre_archivo, mostrar_por_terminal=False):
    try:
        with open(nombre_archivo, 'r', encoding='utf-8') as f:
            lineas = [l.strip() for l in f if l.strip()]

        lineas_desofuscadas = []
        for linea in lineas:
            desofuscado = desofuscar_indicadores(linea)
            # SIEMPRE mostramos ambos
            print(f"Original: {linea}")
            print(f"Desofuscado: {desofuscado}")

            if es_ioc_valido(desofuscado):
                lineas_desofuscadas.append(desofuscado)
                print("✅ VÁLIDO\n")
            else:
                print("❌ NO VÁLIDO\n")

        if mostrar_por_terminal:
            print("\n=== RESULTADOS VÁLIDOS ===")
            for i, ioc in enumerate(lineas_desofuscadas, 1):
                print(f"{i}. {ioc}")
        else:
            archivo_salida = f"resultado_{nombre_archivo}"
            with open(archivo_salida, 'w', encoding='utf-8') as f:
                f.write('\n'.join(lineas_desofuscadas))
            print(f"\n✅ Procesado completo!")
            print(f"📁 Archivo de salida: {archivo_salida}")
            print(f"📊 IOCs válidos: {len(lineas_desofuscadas)}")
            files.download(archivo_salida)

    except Exception as e:
        print(f"❌ Error: {e}")

# Función principal
def main_colab():
    print("\n=== DESOFUSCADOR DE IOC ===")
    print("1. IOC individual")
    print("2. Procesar archivo")
    print("3. Salir")

    opcion = input("\nElige opción (1-3): ").strip()

    if opcion == "1":
        ioc = input("Ingresa IOC: ").strip()
        resultado = desofuscar_indicadores(ioc)
        print(f"\nOriginal: {ioc}")
        print(f"Desofuscado: {resultado}")


    elif opcion == "2":
        print("\nSube tu archivo .txt...")
        uploaded = files.upload()
        for nombre_archivo in uploaded.keys():
            print("\n1. Ver en terminal")
            print("2. Descargar archivo")
            eleccion = input("Elige (1-2): ").strip()
            mostrar = (eleccion == "1")
            procesar_archivo_colab(nombre_archivo, mostrar)

    elif opcion == "3":
        print("👋 Adiós!")
    else:
        print("❌ Opción no válida")

# Ejecutar
main_colab()


=== DESOFUSCADOR DE IOC ===
1. IOC individual
2. Procesar archivo
3. Salir

Elige opción (1-3): 1
Ingresa IOC: hxxp://microsoft-security-update.s3.eu-west-1.amazonaws.com/login.php

Original: hxxp://microsoft-security-update.s3.eu-west-1.amazonaws.com/login.php
Desofuscado: http://microsoft-security-update.s3.eu-west-1.amazonaws.com/login.php


# ***NORMALIZACIÓN***

In [ ]:
import os
import re
import ipaddress
from urllib.parse import urlparse, unquote
from google.colab import files

# Clase para normalizar IoCs
class IOCNormalizer:
    def normalize_ip(self, ip):
        try:
            return str(ipaddress.ip_address(ip.strip()))
        except ValueError:
            return None

    def normalize_domain(self, domain):
        domain = domain.strip().lower()
        if domain.startswith(('http://', 'https://')):
            parsed = urlparse(domain)
            domain = parsed.netloc or parsed.path
        domain = domain.split(':')[0]
        return domain if '.' in domain else None

    def normalize_url(self, url):
        try:
            url = url.strip()
            if not url.startswith(('http://', 'https://')):
                url = 'http://' + url
            parsed = urlparse(url)
            path = unquote(parsed.path)
            query = unquote(parsed.query) if parsed.query else ""
            normalized_url = f"{parsed.scheme}://{parsed.netloc}{path}"
            if query:
                normalized_url += f"?{query}"
            return normalized_url.lower()
        except:
            return None

    def is_valid_ioc(self, ioc):
        try:
            ipaddress.ip_address(ioc.strip())
            return True
        except ValueError:
            pass
        if '.' in ioc:
            return True
        if ioc.startswith(('http://', 'https://')):
            return True
        return False

    def is_obfuscated(self, ioc):
        # Patrones de ofuscación REALES (no URLs ya desofuscadas)
        patterns = [
            r'\[\.\]', r'\[dot\]', r'\(dot\)',
            r'hxxp://', r'hXXp://', r'h\*\*p://',  # Solo cuando tiene ://
            r'htxp://', r'\[://\]'
        ]
        return any(re.search(p, ioc, re.IGNORECASE) for p in patterns)

    def normalize_ioc(self, ioc):
        ioc = ioc.strip()
        if re.match(r'^\d{1,3}(\.\d{1,3}){3}$', ioc):
            return self.normalize_ip(ioc)
        elif ioc.startswith(('http://', 'https://')) or '://' in ioc:
            return self.normalize_url(ioc)
        elif '.' in ioc:
            return self.normalize_domain(ioc)
        return ioc

    def process_file(self, file_path):
        normalized_iocs = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue

                print(f"\n📝 Procesando: {line}")

                if not self.is_valid_ioc(line):
                    print(f"❌ IoC inválido omitido: {line}")
                    continue

                if self.is_obfuscated(line):
                    print(f"⚠️ IoC ofuscado detectado: {line}")
                    continue

                normalized = self.normalize_ioc(line)
                if normalized:
                    print(f"✅ Normalizado: {normalized}")
                    normalized_iocs.append(normalized)
                else:
                    print(f"❌ No se pudo normalizar: {line}")

        return normalized_iocs

    def create_normalized_file(self, normalized_iocs, original_file_path):
        filename_base = os.path.splitext(original_file_path)[0]
        output_filename = f"{filename_base}_normalizado.txt"
        with open(output_filename, 'w', encoding='utf-8') as f:
            for ioc in normalized_iocs:
                f.write(ioc + '\n')
        return output_filename

# Función principal
def main():
    normalizer = IOCNormalizer()

    print("=== NORMALIZADOR DE INDICADORES DE COMPROMISO ===")
    print("\nOpciones:")
    print("1. Normalizar un IOC individual")
    print("2. Subir y procesar un archivo .txt")
    print("3. Salir")

    opcion = input("\nSelecciona una opción (1, 2 o 3): ").strip()

    if opcion == "1":
        ioc = input("\nIngresa el IOC: ").strip()
        print(f"\n🔍 Analizando: {ioc}")

        if not normalizer.is_valid_ioc(ioc):
            print("❌ No puedo normalizar IoC que no sean del tipo IP, URL o dominio.")
            return

        if normalizer.is_obfuscated(ioc):
            print("⚠️ La IoC está ofuscada. Por favor, vuelve al paso 1 y desofúscala.")
            print("Patrones ofuscados detectados:")
            patterns = [r'\[\.\]', r'\[dot\]', r'\(dot\)', r'hxxp://', r'hXXp://', r'h\*\*p://', r'htxp://', r'\[://\]']
            for pattern in patterns:
                if re.search(pattern, ioc, re.IGNORECASE):
                    print(f"  - {pattern}")
            return

        normalizado = normalizer.normalize_ioc(ioc)
        print(f"\n📊 RESULTADO:")
        print(f"Original: {ioc}")
        print(f"Normalizado: {normalizado}")
        return

    elif opcion == "2":
        print("\nSube el archivo .txt con los IoCs...")
        uploaded = files.upload()
        for file_path in uploaded.keys():
            iocs = normalizer.process_file(file_path)

            if not iocs:
                print("\n❌ No se encontraron IoCs válidos para normalizar.")
                return

            print("\n¿Qué deseas hacer con los resultados?")
            print("1. Mostrar por terminal")
            print("2. Descargar archivo normalizado")
            eleccion = input("Selecciona una opción (1 o 2): ").strip()

            if eleccion == '1':
                print("\n=== RESULTADO NORMALIZADO ===")
                for i, ioc in enumerate(iocs, 1):
                    print(f"{i}. {ioc}")
            elif eleccion == '2':
                output_file = normalizer.create_normalized_file(iocs, file_path)
                print(f"\n✅ Archivo '{output_file}' creado exitosamente.")
                print(f"📁 Contiene {len(iocs)} IoCs normalizados")
                files.download(output_file)
            else:
                print("❌ Opción no válida.")
        return

    elif opcion == "3":
        print("👋 Saliendo del programa...")
        return

    else:
        print("❌ Opción no válida.")

# Ejecutar
main()

=== NORMALIZADOR DE INDICADORES DE COMPROMISO ===

Opciones:
1. Normalizar un IOC individual
2. Subir y procesar un archivo .txt
3. Salir

Selecciona una opción (1, 2 o 3): 1

Ingresa el IOC: http://microsoft-security-update.s3.eu-west-1.amazonaws.com/login.php

🔍 Analizando: http://microsoft-security-update.s3.eu-west-1.amazonaws.com/login.php

📊 RESULTADO:
Original: http://microsoft-security-update.s3.eu-west-1.amazonaws.com/login.php
Normalizado: http://microsoft-security-update.s3.eu-west-1.amazonaws.com/login.php


# ***CLASIFICACIÓN IOC***

In [ ]:
import re
import os
from datetime import datetime
from google.colab import files

# Clase para clasificar IoCs
class IOCClassifier:
    def __init__(self):
        self.patterns = {
            'ipv4': r'^(\d{1,3}\.){3}\d{1,3}$',
            'ipv6': r'^([0-9a-fA-F]{1,4}:){7}[0-9a-fA-F]{1,4}$',
            'domain': r'^([a-zA-Z0-9]([a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?\.)+[a-zA-Z]{2,}$',  # CORREGIDO
            'url': r'^(https?|ftp)://[^\s/$.?#].[^\s]*$'
        }

    def classify_ioc(self, ioc):
        ioc = ioc.strip()
        if re.match(self.patterns['url'], ioc, re.IGNORECASE):
            return "URL"
        if re.match(self.patterns['ipv4'], ioc):
            return "IPv4"
        if re.match(self.patterns['ipv6'], ioc):
            return "IPv6"
        if re.match(self.patterns['domain'], ioc):
            return "Dominio"
        return "Desconocido"

# El resto del código se mantiene igual...
def clasificar_individual(classifier):
    print("\n--- CLASIFICACIÓN INDIVIDUAL ---")
    ioc = input("Introduce la IOC a clasificar: ").strip()
    if not ioc:
        print("Error: No se introdujo ninguna IOC.")
        return
    tipo = classifier.classify_ioc(ioc)
    print(f"\nResultado:")
    print(f"IOC: {ioc}")
    print(f"Tipo: {tipo}")

def clasificar_archivo(classifier):
    print("\nSube el archivo .txt con los IoCs...")
    uploaded = files.upload()
    for archivo_entrada in uploaded.keys():
        try:
            with open(archivo_entrada, 'r', encoding='utf-8') as f:
                lineas = f.readlines()

            resultados = []
            for i, linea in enumerate(lineas, 1):
                ioc = linea.strip()
                if ioc:
                    tipo = classifier.classify_ioc(ioc)
                    resultados.append(f"{ioc} - {tipo}")

            print("\n¿Qué deseas hacer con los resultados?")
            print("1. Mostrar por terminal")
            print("2. Descargar archivo clasificado")
            eleccion = input("Selecciona una opción (1 o 2): ").strip()

            if eleccion == '1':
                print("\n=== RESULTADO DE CLASIFICACIÓN ===")
                for r in resultados:
                    print(r)
            elif eleccion == '2':
                nombre_base = os.path.splitext(archivo_entrada)[0]
                archivo_salida = f"{nombre_base}_clasificacion.txt"
                with open(archivo_salida, 'w', encoding='utf-8') as f:
                    f.write("CLASIFICACIÓN DE IOCS\n")
                    f.write("=" * 50 + "\n")
                    f.write(f"Archivo original: {archivo_entrada}\n")
                    f.write(f"Fecha de clasificación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                    f.write("=" * 50 + "\n\n")
                    for resultado in resultados:
                        f.write(resultado + "\n")
                print(f"\n✓ Clasificación completada exitosamente")
                print(f"✓ Resultados guardados en: {archivo_salida}")
                print(f"✓ Total de IOCs procesadas: {len(resultados)}")
                files.download(archivo_salida)
            else:
                print("❌ Opción no válida.")
        except Exception as e:
            print(f"Error al procesar el archivo: {str(e)}")

def main():
    classifier = IOCClassifier()

    print("\n=== CLASIFICADOR DE IOC ===")
    print("1. Clasificar una única IOC")
    print("2. Subir y clasificar archivo .txt")
    print("3. Salir")

    opcion = input("\nSelecciona una opción (1, 2 o 3): ").strip()

    if opcion == '1':
        clasificar_individual(classifier)
    elif opcion == '2':
        clasificar_archivo(classifier)
    elif opcion == '3':
        print("Saliendo del programa...")
    else:
        print("❌ Opción no válida.")

# Ejecutar
main()


=== CLASIFICADOR DE IOC ===
1. Clasificar una única IOC
2. Subir y clasificar archivo .txt
3. Salir

Selecciona una opción (1, 2 o 3): 1

--- CLASIFICACIÓN INDIVIDUAL ---
Introduce la IOC a clasificar: http://microsoft-security-update.s3.eu-west-1.amazonaws.com/login.php

Resultado:
IOC: http://microsoft-security-update.s3.eu-west-1.amazonaws.com/login.php
Tipo: URL


# ***RESOLUCIÓN DNS***

In [ ]:
# Importamos las librerías necesarias
import socket
import ipaddress
from google.colab import files
import os
import re
from datetime import datetime

# Configurar timeout para las operaciones de red
socket.setdefaulttimeout(10)  # 10 segundos de timeout

# Función para detectar si un IoC está ofuscado
def is_obfuscated(ioc):
    obfuscation_patterns = ['[.]', '(.)', 'hxxp', '[at]', '[dot]']  # Patrones comunes de ofuscación
    return any(p in ioc.lower() for p in obfuscation_patterns)  # Retorna True si encuentra alguno

# Función para verificar si un IoC está normalizado
def is_normalized(ioc):
    return ioc == ioc.strip().lower() and ' ' not in ioc  # Debe estar en minúsculas y sin espacios

# Función para limpiar IOC (remover clasificación previa)
def clean_ioc(ioc):
    # Remover clasificaciones como " - Dominio", " - URL", etc.
    patterns_to_remove = [
        r'\s*-\s*Dominio$', r'\s*-\s*URL$', r'\s*-\s*IPv4$',
        r'\s*-\s*IPv6$', r'\s*-\s*Desconocido$'
    ]
    cleaned_ioc = ioc.strip()
    for pattern in patterns_to_remove:
        cleaned_ioc = re.sub(pattern, '', cleaned_ioc, flags=re.IGNORECASE)
    return cleaned_ioc.strip()

# Función para realizar resolución DNS (A, AAAA, PTR) con mejor manejo de errores
def resolve_dns(ioc):
    results = {'IoC': ioc}  # Diccionario para guardar resultados

    try:
        # Verificar si es una dirección IP
        ip = ipaddress.ip_address(ioc)

        # Resolución PTR para IPs
        try:
            ptr_result = socket.gethostbyaddr(str(ip))
            results['PTR'] = ptr_result[0]
        except (socket.herror, socket.gaierror, socket.timeout):
            results['PTR'] = 'No se encontró registro PTR'
        except Exception as e:
            results['PTR'] = f'Error en resolución PTR: {str(e)}'

    except ValueError:
        # Es un dominio/hostname
        try:
            # Resolución A (IPv4)
            a_record = socket.gethostbyname(ioc)
            results['A'] = a_record
        except (socket.gaierror, socket.timeout):
            results['A'] = 'No se encontró dirección IPv4'
        except Exception as e:
            results['A'] = f'Error en resolución A: {str(e)}'

        try:
            # Resolución AAAA (IPv6)
            aaaa_records = socket.getaddrinfo(ioc, None, socket.AF_INET6)
            unique_aaaa = list(set([r[4][0] for r in aaaa_records]))
            results['AAAA'] = unique_aaaa if unique_aaaa else 'No se encontró dirección IPv6'
        except (socket.gaierror, socket.timeout):
            results['AAAA'] = 'No se encontró dirección IPv6'
        except Exception as e:
            results['AAAA'] = f'Error en resolución AAAA: {str(e)}'

    return results

# Función para procesar IoCs desde archivo
def process_iocs_from_file(iocs, filename_base):
    results = []  # Lista para almacenar resultados
    stats = {
        'total': len(iocs),
        'processed': 0,
        'obfuscated': 0,
        'not_normalized': 0,
        'dns_success': 0
    }

    print(f"\n📊 Procesando {stats['total']} IoCs...")

    for i, ioc in enumerate(iocs, 1):
        # Limpiar el IOC de clasificaciones previas
        clean_ioc_value = clean_ioc(ioc)

        if is_obfuscated(clean_ioc_value):
            results.append(f"IoC: {clean_ioc_value} → ⚠️ Ofuscado. Ve al paso 1: Desofuscación.")
            stats['obfuscated'] += 1
            continue

        if not is_normalized(clean_ioc_value):
            results.append(f"IoC: {clean_ioc_value} → ⚠️ No normalizado. Ve al paso 2: Normalización.")
            stats['not_normalized'] += 1
            continue

        stats['processed'] += 1
        dns_info = resolve_dns(clean_ioc_value)

        # Construir resultado
        result = f"IoC: {clean_ioc_value}"
        has_dns_data = False

        for record_type, value in dns_info.items():
            if record_type != 'IoC':
                result += f"\n  {record_type}: {value}"
                if value not in ['No se encontró registro PTR', 'No se encontró dirección IPv4',
                               'No se encontró dirección IPv6'] and not str(value).startswith('Error'):
                    has_dns_data = True

        if has_dns_data:
            stats['dns_success'] += 1

        results.append(result)

        # Mostrar progreso cada 10 IoCs
        if i % 10 == 0 or i == stats['total']:
            print(f"  Procesados: {i}/{stats['total']}")

    # Mostrar estadísticas
    print(f"\n📈 ESTADÍSTICAS:")
    print(f"  Total IoCs: {stats['total']}")
    print(f"  Ofuscados: {stats['obfuscated']}")
    print(f"  No normalizados: {stats['not_normalized']}")
    print(f"  Procesados para DNS: {stats['processed']}")
    print(f"  Con datos DNS: {stats['dns_success']}")

    return results, stats

# Función para mostrar resultados por terminal con paginación
def show_results_terminal(results):
    print("\n" + "="*60)
    print("               RESULTADOS DEL ANÁLISIS DNS")
    print("="*60)

    for i, result in enumerate(results, 1):
        print(f"\n[{i}/{len(results)}] {result}")
        print("-" * 40)

        # Pausar cada 10 resultados para no saturar la terminal
        if i % 10 == 0 and i < len(results):
            print(f"\n--- Mostrados {i} de {len(results)} resultados ---")
            continuar = input("¿Continuar mostrando resultados? (s/n): ").strip().lower()
            if continuar != 's':
                print(f"\n⚠️  Visualización interrumpida. Se mostraron {i} de {len(results)} resultados.")
                break

    print(f"\n✅ Visualización completada. Total mostrados: {min(i, len(results))} de {len(results)}")

# Función para descargar archivo con resultados
def download_results_file(results, filename_base, stats):
    output_filename = f"{filename_base}_DNS_Analysis.txt"

    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("ANÁLISIS DNS DE IOCS\n")
        f.write("=" * 50 + "\n")
        f.write(f"Fecha de análisis: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total IoCs procesados: {stats['total']}\n")
        f.write(f"Ofuscados: {stats['obfuscated']}\n")
        f.write(f"No normalizados: {stats['not_normalized']}\n")
        f.write(f"Procesados para DNS: {stats['processed']}\n")
        f.write(f"Con datos DNS: {stats['dns_success']}\n")
        f.write("=" * 50 + "\n\n")

        for result in results:
            f.write(result + "\n\n")

    files.download(output_filename)
    print(f"✓ Archivo descargado: {output_filename}")

# Función para procesar un IoC individual
def process_single_ioc(ioc):
    clean_ioc_value = clean_ioc(ioc)

    if is_obfuscated(clean_ioc_value):
        print(f"\nIoC: {clean_ioc_value} → ⚠️ Ofuscado. Ve al paso 1: Desofuscación.")
        return
    if not is_normalized(clean_ioc_value):
        print(f"\nIoC: {clean_ioc_value} → ⚠️ No normalizado. Ve al paso 2: Normalización.")
        return

    print(f"\n🔍 Analizando: {clean_ioc_value}")
    dns_info = resolve_dns(clean_ioc_value)

    print(f"\n📋 RESULTADOS PARA: {clean_ioc_value}")
    print("-" * 40)
    for record_type, value in dns_info.items():
        if record_type != 'IoC':
            print(f"  {record_type}: {value}")

# Función para manejar archivos subidos
def handle_uploaded_file():
    print("\n📁 Sube el archivo .txt con los IoCs (uno por línea)...")
    uploaded = files.upload()

    if not uploaded:
        print("❌ No se subió ningún archivo.")
        return None, None

    for filename in uploaded.keys():
        filename_base = os.path.splitext(filename)[0]
        with open(filename, 'r', encoding='utf-8') as f:
            iocs = [line.strip() for line in f if line.strip()]

        print(f"✓ Archivo '{filename}' cargado. Total de IoCs: {len(iocs)}")
        return iocs, filename_base

    return None, None

# Función principal del script
def main():
    while True:
        print("\n" + "="*50)
        print("          ANALIZADOR DNS DE IOCS")
        print("="*50)
        print("¿Cómo deseas introducir los IoCs?")
        print("1. Introducir manualmente")
        print("2. Subir archivo .txt")
        print("3. Salir del programa")

        choice = input("\nSelecciona una opción (1, 2 o 3): ").strip()

        if choice == '1':
            ioc = input("Introduce el IoC: ").strip()
            if ioc:
                process_single_ioc(ioc)
                print("\n👋 Análisis completado. Saliendo del programa...")
                break
            else:
                print("❌ No se introdujo ningún IoC.")

        elif choice == '2':
            iocs, filename_base = handle_uploaded_file()
            if iocs:
                # Procesar los IoCs
                results, stats = process_iocs_from_file(iocs, filename_base)

                # Menú para mostrar resultados
                print("\n" + "="*50)
                print("          OPCIONES DE RESULTADOS")
                print("="*50)
                print("1. Mostrar resultados por terminal")
                print("2. Descargar archivo con resultados")

                result_choice = input("\nSelecciona una opción (1 o 2): ").strip()

                if result_choice == '1':
                    show_results_terminal(results)
                    print("\n👋 Análisis completado. Saliendo del programa...")
                    break

                elif result_choice == '2':
                    download_results_file(results, filename_base, stats)
                    print("\n👋 Análisis completado. Saliendo del programa...")
                    break

                else:
                    print("❌ Opción no válida.")

        elif choice == '3':
            print("\n👋 Saliendo del programa... ¡Hasta pronto!")
            break

        else:
            print("❌ Opción no válida. Por favor, selecciona 1, 2 o 3.")

# Ejecuta la función principal
if __name__ == "__main__":
    main()


          ANALIZADOR DNS DE IOCS
¿Cómo deseas introducir los IoCs?
1. Introducir manualmente
2. Subir archivo .txt
3. Salir del programa

Selecciona una opción (1, 2 o 3): 1
Introduce el IoC: http://microsoft-security-update.s3.eu-west-1.amazonaws.com/login.php

🔍 Analizando: http://microsoft-security-update.s3.eu-west-1.amazonaws.com/login.php

📋 RESULTADOS PARA: http://microsoft-security-update.s3.eu-west-1.amazonaws.com/login.php
----------------------------------------
  A: No se encontró dirección IPv4
  AAAA: No se encontró dirección IPv6

👋 Análisis completado. Saliendo del programa...


# ***INFORME RDAP***

Para esta parte recomiendo poner el archivo.txt que este desofucado y normalizado. A poder ser que no este clasificado ni con la resolución DNS realizada. GRACIAS

In [1]:
# Aquí importo todo lo que necesito para el script.
import requests
import json
import re
import os
from datetime import datetime
from google.colab import files # Específico de Colab


# función principal para consultar la info de un IOC.
def consultar_rdap(ioc):
    try:
        ioc_limpia = ioc.strip()

        if re.match(r'^\d+\.\d+\.\d+\.\d+$', ioc_limpia):
            url = f"https://rdap.apnic.net/ip/{ioc_limpia}"
        elif re.match(r'^https?://', ioc_limpia):
            dominio = re.findall(r'https?://([^/]+)', ioc_limpia)[0]
            url = f"https://rdap.verisign.com/com/v1/domain/{dominio}"
        else:
            dominio = ioc_limpia.split(' - ')[0].strip()
            url = f"https://rdap.verisign.com/com/v1/domain/{dominio}"

        print(f"Consultando información RDAP para: {ioc_limpia}")

        # Hago la petición con un timeout de 10 segundos
        response = requests.get(
            url,
            headers={'Accept': 'application/rdap+json'},
            timeout=10
        )

        response.raise_for_status()
        return response.json()

    # Capturo cualquier error
    except Exception as e:
        print(f"❌ Error en la consulta RDAP: {e}")
        return None

# función para sacar las fechas de creación y expiración.
def extraer_fechas_eventos(datos_rdap):
    fecha_creacion = None
    fecha_expiracion = None

    if 'events' in datos_rdap:
        for evento in datos_rdap['events']:
            if evento.get('eventAction') == 'registration':
                fecha_creacion = evento.get('eventDate')
            elif evento.get('eventAction') == 'expiration':
                fecha_expiracion = evento.get('eventDate')

    return fecha_creacion, fecha_expiracion

# Esta función solo formatea la salida para que se vea bien en la consola.
def mostrar_resultado_individual(ioc, datos_rdap):
    print(f"\n{'='*50}")
    print(f"RESULTADO PARA: {ioc}")
    print(f"{'='*50}")
    print(f"Handle: {datos_rdap.get('handle', 'N/A')}")
    print(f"Nombre: {datos_rdap.get('name', 'N/A')}")

    if 'startAddress' in datos_rdap and 'endAddress' in datos_rdap:
        print(f"Rango IP: {datos_rdap['startAddress']} - {datos_rdap['endAddress']}")

    print(f"Estatus: {', '.join(datos_rdap.get('status', ['N/A']))}")

    fecha_creacion, fecha_expiracion = extraer_fechas_eventos(datos_rdap)
    if fecha_creacion:
        print(f"Fecha de creación: {fecha_creacion}")
    if fecha_expiracion:
        print(f"Fecha de expiración: {fecha_expiracion}")

    entities = datos_rdap.get('entities', [])
    if entities:
        print(f"\nEntidades encontradas: {len(entities)}")

    events = datos_rdap.get('events', [])
    if events:
        print(f"Eventos registrados: {len(events)}")
        for event in events[:2]:
            print(f" - {event.get('eventAction', 'N/A')}: {event.get('eventDate', 'N/A')}")


# La lógica para cuando elijo la opción 1 (analizar un solo IOC).
def analizar_ioc_individual():
    print("\n" + "="*40)
    print(" ANÁLISIS INDIVIDUAL DE IOC")
    print("="*40)
    ioc = input("Introduzca la IOC: ").strip()

    if not ioc:
        print("❌ No se introdujo ninguna IOC.")
        return

    datos_rdap = consultar_rdap(ioc)

    if datos_rdap:
        mostrar_resultado_individual(ioc, datos_rdap)
    else:
        print("❌ No se pudo obtener información RDAP para esta IOC.")


# La lógica para la opción 2 (subir un archivo .txt con muchos IOCs).
def analizar_archivo_ioc():
    print("\nSube el archivo .txt con las IoCs...")
    # Uso la función de Colab para subir el archivo
    uploaded = files.upload()

    for archivo in uploaded.keys():
        with open(archivo, 'r', encoding='utf-8') as f:
            lineas = f.readlines()

        resultados = [] # Guardo todos los resultados aquí

        # Recorro el archivo línea por línea
        for linea in lineas:
            ioc = linea.strip()
            if not ioc:
                continue

            datos_rdap = consultar_rdap(ioc)
            fecha_creacion, fecha_expiracion = extraer_fechas_eventos(datos_rdap) if datos_rdap else (None, None)
            resultados.append((ioc, datos_rdap, fecha_creacion, fecha_expiracion))

        # Doy a elegir qué hacer con los resultados
        print("\n¿Qué deseas hacer con los resultados?")
        print("1. Mostrar por terminal")
        print("2. Descargar archivo con resultados")
        eleccion = input("Selecciona una opción (1 o 2): ").strip()

        if eleccion == '1':
            # Opción 1: Muestro todo en la consola
            for ioc, datos, fc, fe in resultados:
                if datos:
                    mostrar_resultado_individual(ioc, datos)
                else:
                    print(f"\n❌ No se pudo obtener información para: {ioc}")

        elif eleccion == '2':
            # Opción 2: Genero un archivo de reporte
            archivo_salida = archivo.replace('.txt', '_analisisRDAP.txt')
            with open(archivo_salida, 'w', encoding='utf-8') as f:
                # Escribo una cabecera
                f.write("INFORME RDAP - ANÁLISIS DE IOCS\n")
                f.write("=" * 50 + "\n")
                f.write(f"Archivo original: {archivo}\n")
                f.write(f"Fecha de análisis: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write("=" * 50 + "\n\n")

                # Escribo los resultados en el archivo
                for ioc, datos, fc, fe in resultados:
                    f.write(f"IOC: {ioc}\n")
                    if datos:
                        f.write(f"Handle: {datos.get('handle', 'N/A')}\n")
                        f.write(f"Nombre: {datos.get('name', 'N/A')}\n")
                        if 'startAddress' in datos and 'endAddress' in datos:
                            f.write(f"Rango IP: {datos['startAddress']} - {datos['endAddress']}\n")
                        f.write(f"Estatus: {', '.join(datos.get('status', ['N/A']))}\n")
                        if fc:
                            f.write(f"Fecha creación: {fc}\n")
                        if fe:
                            f.write(f"Fecha expiración: {fe}\n")
                    else:
                        f.write("Estado: ERROR - No se pudo obtener información RDAP\n")
                    f.write("-" * 40 + "\n")

            print(f"\n✓ Archivo guardado como: {archivo_salida}")
            # Uso la función de Colab para descargar el reporte
            files.download(archivo_salida)
        else:
            print("❌ Opción no válida.")


# Esta es la función principal que muestra el menú.
def main():
    print("🚀 Iniciando Analizador de IOC...")
    print("\n" + "="*60)
    print(" ANALIZADOR DE IOC")
    print("="*60)
    print("1. Análisis individual de IOC")
    print("2. Subir y analizar archivo de IOC")
    print("3. Salir")
    print("="*60)

    opcion = input("Seleccione una opción (1-3): ").strip()

    if opcion == '1':
        analizar_ioc_individual()
    elif opcion == '2':
        analizar_archivo_ioc()
    elif opcion == '3':
        print("👋 ¡Hasta pronto!")
    else:
        print("❌ Opción no válida.")


# Aquí llamo a la función principal para que todo empiece.
main()

🚀 Iniciando Analizador de IOC...

 ANALIZADOR DE IOC
1. Análisis individual de IOC
2. Subir y analizar archivo de IOC
3. Salir
Seleccione una opción (1-3): 3
👋 ¡Hasta pronto!
